In [0]:
staging_assignment_domain=dbutils.widgets.get("staging_assignment_domain")
assignment_domain=dbutils.widgets.get("assignment_domain")
office=dbutils.widgets.get("office")
payor=dbutils.widgets.get("payor")
date=dbutils.widgets.get("date")
volume_path=dbutils.widgets.get("volume_path")
control_table=dbutils.widgets.get("control_table")

In [0]:
%pip install openpyxl

In [0]:
import re
import os
import pandas as pd
from io import BytesIO
from datetime import datetime

def to_snake_case(name):
    """Convert column names to snake_case format"""
    name = name.strip().replace(' ', '_')
    name = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub('([a-z0-9])([A-Z])', r'\1_\2', name)
    name = name.lower()
    name = re.sub('_+', '_', name)
    return name

def extract_file_metadata(file_path):
    """Extract metadata from file path"""
    file_name = os.path.basename(file_path)
    
    try:
        file_info = dbutils.fs.ls(file_path)[0]
        file_size = file_info.size
    except:
        file_size = 0
    
    return {
        'file_name': file_name,
        'file_path': file_path,
        'file_size': file_size
    }

def is_file_already_processed(file_name, control_table):
    """Check if file has already been processed"""
    try:
        result = spark.sql(f"""
            SELECT COUNT(*) as count 
            FROM {control_table}
            WHERE file_name = '{file_name}' 
            AND status = 'SUCCESS'
        """).collect()[0]['count']
        
        return result > 0
    except Exception as e:
        print(f"Control table check failed: {e}")
        return False

def log_file_processing(file_metadata, record_count, control_table, status='SUCCESS'):
    """Log file processing to control table"""
    try:
        job_run_id = dbutils.notebook.entry_point.getDbutils().notebook().getContext().currentRunId().toString()
    except:
        job_run_id = "unknown"
    
    file_name_escaped = file_metadata['file_name'].replace("'", "''")
    file_path_escaped = file_metadata['file_path'].replace("'", "''")
    
    insert_sql = f"""
    INSERT INTO {control_table} 
    VALUES (
        '{file_name_escaped}',
        '{file_path_escaped}',
        {file_metadata['file_size']},
        {record_count},
        current_timestamp(),
        '{job_run_id}',
        '{status}'
    )
    """
    
    spark.sql(insert_sql)

def get_latest_excel_file(volume_path):
    """Get the most recently modified Excel file in the volume"""
    try:
        files = dbutils.fs.ls(volume_path)
        
        # Filter for Excel files only
        excel_files = [f for f in files if f.path.endswith('.xlsx') or f.path.endswith('.xls')]
        
        if not excel_files:
            raise FileNotFoundError(f"No Excel files found in {volume_path}")
        
        # Sort by modification time (most recent first)
        latest_file = sorted(excel_files, key=lambda x: x.modificationTime, reverse=True)[0]
        
        return latest_file.path
    except Exception as e:
        print(f"Error getting latest Excel file: {e}")
        raise

In [0]:
triggered_file = get_latest_excel_file(volume_path)
file_metadata = extract_file_metadata(triggered_file)

print(f"\nTriggered File: {file_metadata['file_name']}")
print(f"File Path: {file_metadata['file_path']}")
print(f"File Size: {file_metadata['file_size']:,} bytes")

if is_file_already_processed(file_metadata['file_name'], control_table):
    print(f"\n⚠️  FILE ALREADY PROCESSED")
    print(f"File '{file_metadata['file_name']}' has already been processed.")
    print("Skipping to prevent duplicates.")
    print("=" * 80)
    
    last_processing = spark.sql(f"""
        SELECT processing_timestamp, records_processed, job_run_id
        FROM {control_table}
        WHERE file_name = '{file_metadata['file_name']}'
        ORDER BY processing_timestamp DESC
        LIMIT 1
    """).collect()[0]
    
    print(f"\nLast Processed: {last_processing['processing_timestamp']}")
    print(f"Records Processed: {last_processing['records_processed']}")
    print(f"Job Run ID: {last_processing['job_run_id']}")
 
    dbutils.notebook.exit("File already processed - skipped")
else:
    print(f"\n✓ NEW FILE DETECTED")
    print("Proceeding with processing...")
  

In [0]:
try:
    # Clean up file path
    excel_file_path = triggered_file
    if excel_file_path.startswith('dbfs:'):
        excel_file_path = excel_file_path.replace('dbfs:', '')
    
    print(f"\n✓ Found: {file_metadata['file_name']}")
    binary_df = spark.read.format("binaryFile").load(excel_file_path)
    file_content = binary_df.collect()[0]['content']
    
    pdf = pd.read_excel(BytesIO(file_content), sheet_name=0, engine='openpyxl')
    print(f"✓ Read Excel: {len(pdf)} rows, {len(pdf.columns)} columns")
    
    record_count = len(pdf)
 
    pdf = pdf.astype(str)
    df_assignment_domain = spark.createDataFrame(pdf)
    original_columns = df_assignment_domain.columns
    renamed_columns = [to_snake_case(col) for col in original_columns]
    df_assignment_domain_renamed = df_assignment_domain.toDF(*renamed_columns)
    
    # Create view
    df_assignment_domain_renamed.createOrReplaceTempView("assignment_domain_view")
    
    print(f"✓ Created view: assignment_domain_view")
    print("\nColumn transformations:")
    for old, new in zip(original_columns[:5], renamed_columns[:5]):  # Show first 5
        if old != new:
            print(f"  '{old}' → '{new}'")
    if len(original_columns) > 5:
        print(f"  ... and {len(original_columns) - 5} more")
    
    print("\n" + "=" * 80)
    print("DATA PREVIEW")
    print("=" * 80)
    display(spark.sql("SELECT * FROM assignment_domain_view LIMIT 10"))
    print("\n✓ SUCCESS!")
    
except Exception as e:
    print(f"\n❌ ERROR READING EXCEL FILE: {e}")
    log_file_processing(file_metadata, 0, control_table, status='FAILED')
    raise

In [0]:
try:
    spark.sql(f"TRUNCATE TABLE {staging_assignment_domain}")
    print(f"✓ Truncated: {staging_assignment_domain}")
    
except Exception as e:
    print(f"❌ ERROR TRUNCATING TABLE: {e}")
    log_file_processing(file_metadata, record_count, control_table, status='FAILED')
    raise

In [0]:
try:
    df_assignment_domain.write.mode("append").insertInto(staging_assignment_domain)
    print(f"✓ Inserted {record_count:,} records into: {staging_assignment_domain}")
    
except Exception as e:
    print(f"❌ ERROR INSERTING INTO STAGING: {e}")
    log_file_processing(file_metadata, record_count, control_table, status='FAILED')
    raise

In [0]:
try:
    count_before = spark.sql(f"SELECT COUNT(*) as cnt FROM {assignment_domain}").collect()[0]['cnt']
    spark.sql(f"""
    INSERT INTO {assignment_domain} (
        reporting_week_ending_date_key,
        office_key,
        payor_key,
        bill_medium,
        billing_period,
        collector,
        supervisor,
        follow_up,
        workday_follow_up,
        practice,
        authorization_team,
        confirmation_team,
        authorization_coordinator,
        confirmation_coordinator
    )
    SELECT
        CAST(replace(CAST(date_sub(dt.WeekEndingDate, 7) AS STRING), '-', '') AS INT) AS reporting_week_ending_date_key,
        ofc.OfficeKey as office_key,
        pyr.PayorKey as payor_key,
        ad.bill_medium as bill_medium,
        ad.billing_period as billing_period,
        ad.collector as collector,
        ad.supervisor as supervisor,
        ad.follow_up as follow_up,
        ad.workday_follow_up as workday_follow_up,
        ad.practice as practice,
        ad.authorization_team as authorization_team,
        ad.confirmation_team as confirmation_team,
        ad.authorization_coordinator as authorization_coordinator,
        ad.confirmation_coordinator as confirmation_coordinator
    FROM {staging_assignment_domain} ad
    INNER JOIN {office} ofc
        ON ad.oabbr = ofc.OfficeAbbreviation
    INNER JOIN {payor} pyr
        ON ad.payor_id = pyr.PayorID
    INNER JOIN {date} dt
        ON dt.CalendarDate = current_date()
    """)
    
    count_after = spark.sql(f"SELECT COUNT(*) as cnt FROM {assignment_domain}").collect()[0]['cnt']
    records_inserted = count_after - count_before
    
    print(f"✓ Inserted {records_inserted:,} records into: {assignment_domain}")
    log_file_processing(file_metadata, records_inserted, control_table, status='SUCCESS')
    
    print("\n✓ Processing logged to control table")
except Exception as e:
    print(f"\n❌ ERROR INSERTING INTO FACT TABLE: {e}")
    log_file_processing(file_metadata, record_count, control_table, status='FAILED')
    raise